In [ ]:
import cv2
import numpy as np

capture = cv2.VideoCapture("input.mp4")
if not capture.isOpened():
    raise RuntimeError("Cannot open video file")

# Read first frame
ret, old_frame = capture.read()
if not ret or old_frame is None:
    raise RuntimeError("Cannot read first frame")

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# 1) Choose points to track (corners) using Shi-Tomasi
feature_params = dict(
    maxCorners=200,        # maximum number of points
    qualityLevel=0.3,      # higher = fewer but stronger corners
    minDistance=7,         # minimum distance between corners
    blockSize=7
)

p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
if p0 is None:
    raise RuntimeError("No features found to track in the first frame")

# 2) Lucas-Kanade optical flow parameters
lk_params = dict(
    winSize=(15, 15),      # search window size
    maxLevel=2,            # pyramid levels
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
)

# Create a mask image for drawing tracks
mask = np.zeros_like(old_frame)

while True:
    ret, frame = capture.read()
    if not ret or frame is None:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 3) Calculate optical flow (Lucas-Kanade)
    p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

    # If tracking failed completely, stop
    if p1 is None or st is None:
        break

    # 4) Select good points (st==1 means successfully tracked)
    good_new = p1[st.flatten() == 1]
    good_old = p0[st.flatten() == 1]

    # 5) Draw the tracks
    for (new, old) in zip(good_new, good_old):
        a, b = new.ravel()
        c, d = old.ravel()

        # line from old -> new
        mask = cv2.line(mask, (int(c), int(d)), (int(a), int(b)), (0, 255, 0), 2)
        # circle at new point
        frame = cv2.circle(frame, (int(a), int(b)), 3, (0, 0, 255), -1)

    output = cv2.add(frame, mask)
    cv2.imshow("Lucas-Kanade Optical Flow (Sparse)", output)

    k = cv2.waitKey(20) & 0xFF
    if k == ord('e'):
        break
    elif k == ord('r'):
        # Refresh/re-detect features if tracking drifts
        p0 = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
        mask = np.zeros_like(old_frame)
        old_gray = frame_gray.copy()
        continue
    elif k == ord('s'):
        cv2.imwrite("lucas_frame.png", frame)
        cv2.imwrite("lucas_tracks.png", output)

    # 6) Update for next loop
    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)

capture.release()
cv2.destroyAllWindows()